# netstead quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/e-lo/netstead/blob/main/packages/netstead/docs/notebooks/quickstart.ipynb)

Five cells, end-to-end: install, load a GMNS network straight from a public URL,
validate it, render the map+table HTML viewer inline, and drill into a single
link via ibis.

Run in [Colab](https://colab.research.google.com/) or any Jupyter environment.
No local files needed: the Leavenworth, WA example network (339 links, 121 nodes)
is fetched over HTTPS from this repo as a zipped-CSV bundle. The same network
also ships inside the `netstead` package as `netstead.fixtures.leavenworth`.

In [ ]:
# 1. Install netstead with the [reports] extra (the map+table HTML viewer).
#    netstead 1.0.0b2 is not on PyPI yet (the packages were renamed from gmnspy /
#    datagrove in October 2026 and the first release under the new names is
#    pending), so install both packages from GitHub. Takes about a minute.
CORRAL = "dbcorral @ git+https://github.com/e-lo/netstead.git@main#subdirectory=packages/corral"
NETSTEAD = "netstead[reports] @ git+https://github.com/e-lo/netstead.git@main#subdirectory=packages/netstead"
%pip install -q "{CORRAL}" "{NETSTEAD}"

In [ ]:
# 2. Load a GMNS network from a URL — no clone, no local files.
#    A zipped-CSV bundle is one file, so the remote reader can fetch it directly.
#    (Offline alternative: `from netstead.fixtures import leavenworth` and pass
#    `leavenworth.csv_dir()` instead of the URL.)
from netstead import Network

FIXTURE = "https://raw.githubusercontent.com/e-lo/netstead/main/packages/netstead/netstead/fixtures/leavenworth/leavenworth.csv.zip"
net = Network.from_source(FIXTURE)
net  # _repr_html_ shows GMNS spec version, table inventory, link/node counts

In [ ]:
# 3. Validate against the embedded GMNS spec (the four-pass validator:
#    structural / schema / foreign-key / sync-state). Leavenworth has no ERRORs;
#    expect no WARNINGs and 16 INFO
#    findings (one per optional GMNS table the fixture doesn't ship).
report = net.validate()
report  # _repr_html_ shows per-severity counts + the first issues

In [ ]:
# 4. Render the interactive map+table viewer and open it inline.
from IPython.display import IFrame
from netstead.map import render_validation_html

out_path = "leavenworth_report.html"
with open(out_path, "w") as f:
    f.write(render_validation_html(net, report))
IFrame(out_path, width=900, height=600)

In [ ]:
# 5. Drill into a single link via ibis (no need to materialize the whole table).
first_link_id = int(net.links.to_pandas().iloc[0]["link_id"])
net.links.filter(lambda t: t.link_id == first_link_id).to_pandas()